# 03 — Decoupling Scores

Quantify how selectively a PXR target gene is coupled in hepatocytes relative to every other cell type.

**Input**: `data/processed/coupling.csv`  
**Output**: `data/processed/decoupling.csv`

### Definition

$$\text{DS}_{g,c} = \rho_{\text{hepatocyte}, g} - \rho_{c, g}$$

A high DS for gene *g* in cell type *c* means the gene is strongly coupled to PXR in hepatocytes but not in *c*. The mean DS across all non-hepatocyte cell types ranks genes by their hepatocyte selectivity.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from pxr_uncoupling.config import COLOR_ACCENT, COLOR_CREAM, DATA_PROCESSED
from pxr_uncoupling.coupling import decoupling_score

## Load coupling matrix

In [ ]:
coupling = pd.read_csv(DATA_PROCESSED / "coupling.csv", index_col=0)
print(f"Coupling matrix: {coupling.shape}")
display(coupling.round(3))

## Compute decoupling scores

In [ ]:
ds = decoupling_score(coupling, reference_cell_type="hepatocyte")
print(f"Decoupling matrix: {ds.shape}  (cell_type × gene, relative to hepatocyte)")
display(ds.round(3))

## Save

In [ ]:
out = DATA_PROCESSED / "decoupling.csv"
ds.to_csv(out)
print(f"Saved → {out}")

## Gene ranking by mean decoupling score

In [ ]:
gene_rank = ds.mean(axis=0).sort_values(ascending=False)
print("Top 10 hepatocyte-selective PXR target genes:")
display(gene_rank.head(10).to_frame("mean_DS").round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
fig.patch.set_facecolor(COLOR_CREAM)
ax.set_facecolor(COLOR_CREAM)

colors = [COLOR_ACCENT if v > 0 else "#8a9a7b" for v in gene_rank.values]
gene_rank.plot.barh(ax=ax, color=colors, edgecolor="#d0c4b0")
ax.axvline(0, color="#555", lw=1, ls="--")
ax.set_xlabel("Mean decoupling score (ρ_hepatocyte − ρ_other)")
ax.set_title("PXR target gene hepatocyte selectivity")
plt.tight_layout()
plt.show()

## Per-gene DS across cell types

Visualise where the selectivity comes from for the top 5 genes.

In [ ]:
top5 = gene_rank.head(5).index.tolist()

fig, axes = plt.subplots(1, len(top5), figsize=(3 * len(top5), 4), sharey=True)
fig.patch.set_facecolor(COLOR_CREAM)

for ax, gene in zip(axes, top5):
    ax.set_facecolor(COLOR_CREAM)
    vals = ds[gene].sort_values(ascending=True)
    ax.barh(range(len(vals)), vals.values, color=COLOR_ACCENT, edgecolor="#d0c4b0")
    ax.set_yticks(range(len(vals)))
    ax.set_yticklabels(vals.index, fontsize=7)
    ax.axvline(0, color="#555", lw=0.8, ls="--")
    ax.set_title(gene, fontsize=9)
    ax.set_xlabel("DS", fontsize=8)

fig.suptitle("Decoupling score per cell type — top 5 hepatocyte-selective genes", fontsize=10)
plt.tight_layout()
plt.show()

## Coupling vs. decoupling scatter

Genes in the upper-right quadrant are strongly coupled in hepatocytes AND strongly decoupled elsewhere — the most attractive hepatocyte-selective targets.

In [ ]:
hep_rho = coupling.loc["hepatocyte"]
mean_ds = ds.mean(axis=0)

fig, ax = plt.subplots(figsize=(7, 5))
fig.patch.set_facecolor(COLOR_CREAM)
ax.set_facecolor(COLOR_CREAM)

ax.scatter(hep_rho, mean_ds, color=COLOR_ACCENT, alpha=0.8, s=60, edgecolors="none")
for gene in hep_rho.index:
    ax.annotate(gene, (hep_rho[gene], mean_ds[gene]), fontsize=7,
                xytext=(3, 3), textcoords="offset points")

ax.axhline(0, color="#888", lw=0.8, ls="--")
ax.axvline(0, color="#888", lw=0.8, ls="--")
ax.set_xlabel("Hepatocyte coupling ρ")
ax.set_ylabel("Mean decoupling score")
ax.set_title("Hepatocyte coupling vs. selectivity")
plt.tight_layout()
plt.show()